In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
from tqdm import tqdm

# Descargas necesarias de NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Habilitar la barra de progreso para pandas
tqdm.pandas(desc="Preprocesando textos")

[nltk_data] Downloading package punkt to C:\Users\HP
[nltk_data]     LAUTARO\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\HP
[nltk_data]     LAUTARO\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\HP
[nltk_data]     LAUTARO\AppData\Roaming\nltk_data...


In [2]:
class TextPreprocessor:
    def __init__(self, language='english'):
        self.stop_words = set(stopwords.words(language))
        self.lemmatizer = WordNetLemmatizer()

    def clean_text(self, text):
        text = str(text).lower()
        # Conservar solo letras (inglés) y espacios
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize_and_lemmatize(self, text):
        tokens = word_tokenize(text)
        processed_tokens = [
            self.lemmatizer.lemmatize(word) 
            for word in tokens 
            if word not in self.stop_words
        ]
        return ' '.join(processed_tokens)

    def process_pipeline(self, text):
        cleaned = self.clean_text(text)
        return self.tokenize_and_lemmatize(cleaned)

In [3]:
# 1. Cargar datos
print("Cargando dataset...")
df = pd.read_csv('../data/raw/amazon_reviews_raw.csv')

# 2. Inicializar preprocesador
preprocessor = TextPreprocessor(language='english')

# 3. Aplicar limpieza con barra de progreso (progress_apply en lugar de apply)
df['processed_text'] = df['text'].progress_apply(preprocessor.process_pipeline)

# 4. Eliminar filas que hayan quedado vacías tras la limpieza
df = df.dropna(subset=['processed_text'])
df = df[df['processed_text'].str.strip() != '']

# 5. Guardar resultado
df.to_csv('../data/processed/amazon_reviews_processed.csv', index=False)
print("¡Preprocesamiento completado y guardado!")

# Ver el resultado
df[['text', 'processed_text']].head()

Cargando dataset...


Preprocesando textos: 100%|██████████| 210000/210000 [01:04<00:00, 3251.75it/s]


¡Preprocesamiento completado y guardado!


,text,processed_text
0,These are good tights. They've held up for sev...,good tights held several month brighter pink t...
1,Beautiful I got this to give to someone.,beautiful got give someone
2,I was not expecting to have to jam the power c...,expecting jam power cord behind trim audi goin...
3,Battery had charging problems from day 1. At f...,battery charging problem day first fully charg...
4,I loved the Ann Cleeves books set in Shetland ...,loved ann cleeves book set shetland anticipate...
